# Chào mừng đến Lab Day 2!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Ngay trước khi bắt đầu —</h2>
            <span style="color:#f71;">Tôi muốn dành một chút để chỉ bạn tới trang tài nguyên hữu ích cho khóa học. Trang này gồm liên kết tới tất cả các slides (bài trình chiếu).<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Hãy bookmark (đánh dấu) trang này, và tôi sẽ tiếp tục thêm các liên kết hữu ích theo thời gian.
            </span>
        </td>
    </tr>
</table>

## Trước hết — hãy nói về Chat Completions API (API hoàn thiện hội thoại)

1. Cách đơn giản nhất để gọi một LLM (mô hình ngôn ngữ lớn)
2. Được gọi là Chat Completions vì ý nghĩa là: “đây là một cuộc hội thoại, hãy dự đoán phần tiếp theo nên là gì”
3. Chat Completions API do OpenAI phát minh, nhưng phổ biến đến mức ai cũng dùng!

### Chúng ta sẽ bắt đầu bằng cách gọi OpenAI lần nữa — đừng lo nếu bạn không dùng OpenAI, tới lượt bạn ngay!


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


## Bạn có biết Endpoint (điểm cuối API) là gì không?

Nếu chưa, hãy xem lại hướng dẫn Technical Foundations (nền tảng kỹ thuật) trong thư mục guides

Và đây là một endpoint có thể khiến bạn quan tâm...

In [ ]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

In [ ]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

In [ ]:
response.json()["choices"][0]["message"]["content"]

# Package openai là gì?

Nó được gọi là một Python Client Library (thư viện client Python).

Nó không gì hơn một wrapper (lớp bọc) quanh việc gọi chính xác HTTP endpoint (điểm cuối HTTP) này.

Nó chỉ cho phép bạn làm việc với code Python gọn gàng thay vì phải vật lộn với các đối tượng json lằng nhằng.

Chỉ vậy thôi. Nó là open-source (mã nguồn mở) và lightweight (nhẹ). Một số người nghĩ nó chứa code model của OpenAI — không phải vậy!


In [ ]:
# Tạo OpenAI client (đối tượng client OpenAI)

from openai import OpenAI
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content



## Rồi điều tuyệt này đã xảy ra:

Chat Completions API của OpenAI quá phổ biến, đến mức các nhà cung cấp model khác tạo ra các endpoint (điểm cuối API) giống hệt.

Chúng được gọi là "OpenAI Compatible Endpoints" (endpoint tương thích OpenAI).

Ví dụ, Google làm một cái ở đây: https://generativelanguage.googleapis.com/v1beta/openai/

Và OpenAI quyết định tử tế: họ nói, này, bạn cứ dùng cùng client library (thư viện client) mà chúng tôi làm cho GPT. Chúng tôi cho phép bạn chỉ định một URL endpoint khác và một key (khóa) khác, để dùng nhà cung cấp khác.

Vậy bạn có thể dùng:

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

Và để rõ ràng — dù OpenAI xuất hiện trong code, chúng ta chỉ dùng Python client library nhẹ này để gọi endpoint — không có model OpenAI nào tham gia ở đây.

Nếu bạn thấy rối, hãy xem lại Guide 9 trong thư mục Guides!

Và bây giờ hãy thử!

## PHẦN NÀY LÀ TÙY CHỌN — nhưng nếu muốn thử Google Gemini, hãy vào:

https://aistudio.google.com/

Và tạo API key tại

https://aistudio.google.com/api-keys

Rồi thêm key vào file `.env`, nhớ Save (lưu) file `.env` sau khi sửa:

`GOOGLE_API_KEY=AIz...`


In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith(("AIz", "AQ.")):
    print("An API key was found, but it doesn't start with AIz or AQ.")
else:
    print("API key found and looks good so far!")



In [ ]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

## Và Ollama cũng cung cấp một OpenAI compatible endpoint (endpoint tương thích OpenAI)

...và nó chạy trên máy local (máy của bạn)!

Nếu ô tiếp theo không in ra "Ollama is running" thì hãy mở terminal (cửa sổ dòng lệnh) và chạy `ollama serve`

In [ ]:
requests.get("http://localhost:11434").content

### Tải llama3.2 từ Meta

Đổi thành llama3.2:1b nếu máy của bạn yếu hơn.

Đừng dùng llama3.3 hoặc llama4! Chúng quá lớn với máy của bạn..

In [ ]:
!ollama pull llama3.2

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# Lấy một fun fact (sự thật thú vị)

response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

In [ ]:
# Bây giờ hãy thử deepseek-r1:1.5b — đây là DeepSeek được "distilled" (chưng cất/nén kiến thức) vào Qwen của Alibaba Cloud

!ollama pull deepseek-r1:1.5b

In [ ]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

# BÀI TẬP VỀ NHÀ

Nâng cấp dự án day 1 (tóm tắt webpage / trang web) để dùng một Open Source model (mô hình mã nguồn mở) chạy local (trên máy bạn) qua Ollama thay vì OpenAI

Bạn sẽ dùng được kỹ thuật này cho mọi dự án sau nếu muốn không dùng paid APIs (API trả phí).

**Lợi ích:**
1. Không mất phí API — open-source (mã nguồn mở)
2. Data (dữ liệu) không rời khỏi máy bạn

**Nhược điểm:**
1. Yếu hơn đáng kể so với Frontier Model (mô hình tiên phong)

## Tóm tắt lại cách cài Ollama

Chỉ cần vào [ollama.com](https://ollama.com) và cài!

Sau khi xong, ollama server (máy chủ Ollama) lẽ ra đã chạy local.  
Nếu bạn vào:  
[http://localhost:11434/](http://localhost:11434/)

Bạn sẽ thấy dòng `Ollama is running`.  

Nếu không, mở Terminal mới (Mac) hoặc Powershell (Windows) rồi nhập `ollama serve`  
Và ở một Terminal (Mac) hoặc Powershell (Windows) khác, nhập `ollama pull llama3.2`  
Rồi thử lại [http://localhost:11434/](http://localhost:11434/).

Nếu Ollama chạy chậm trên máy bạn, hãy thử `llama3.2:1b` làm lựa chọn thay thế. Chạy `ollama pull llama3.2:1b` từ Terminal hoặc Powershell, và đổi code từ `MODEL = "llama3.2"` thành `MODEL = "llama3.2:1b"`